# 03d — SVD and Friends

## The One-Sentence Version
PCA is one member of a family — SVD, factor analysis, and ICA each make different assumptions and serve different purposes.

## What You'll Build Intuition For
- The relationship between PCA and SVD
- When factor analysis is more appropriate than PCA
- How ICA finds independent (not just uncorrelated) components
- A decision framework for choosing the right method

## Prerequisites
- [03b — PCA From Scratch](03b_pca_from_scratch.ipynb) — how PCA works internally
- [03c — PCA Applied](03c_pca_applied.ipynb) — PCA on real data

## The Story

PCA finds directions of maximum variance. But variance isn't always what you want.

Sometimes you want the mathematically equivalent but numerically superior SVD. Sometimes you believe your data was generated by a few latent factors plus noise — that's factor analysis. And sometimes you want to unmix signals that are statistically independent, not just uncorrelated — that's ICA.

These methods are siblings, not competitors. Each makes a different assumption about the data's structure, and each shines in a different context. By the end of this notebook, you'll know when to reach for which.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA, FastICA, FactorAnalysis

from utils.plotting import apply_style, COLOURS, PALETTE
from utils.data_generators import make_digit_data

apply_style()
rng = np.random.default_rng(42)

## SVD: PCA's More General Cousin

The Singular Value Decomposition factorises any matrix $X$ as:

$$X = U \Sigma V^T$$

- $U$: left singular vectors (n × n) — patterns across samples
- $\Sigma$: singular values (diagonal) — importance of each pattern
- $V^T$: right singular vectors (d × d) — patterns across features

The connection to PCA: the columns of $V$ are the eigenvectors of $X^T X$ — the same as PCA's principal components. The singular values $\sigma_i$ relate to eigenvalues by $\lambda_i = \sigma_i^2 / (n-1)$.

In [ ]:
# Generate some data and compute PCA vs SVD
data, target, images = make_digit_data()
X = data - data.mean(axis=0)  # centre
n, d = X.shape

# Method 1: PCA via eigendecomposition of covariance
C = np.cov(X.T)
eig_vals, eig_vecs = np.linalg.eigh(C)
idx = np.argsort(eig_vals)[::-1]
eig_vals = eig_vals[idx]
eig_vecs = eig_vecs[:, idx]

# Method 2: SVD directly on the data matrix
U, S, Vt = np.linalg.svd(X, full_matrices=False)

# Compare: right singular vectors should match eigenvectors
# (up to sign — eigenvector sign is arbitrary)
print("SVD singular values vs PCA eigenvalues:")
print(f"{'Component':>10} {'σ (SVD)':>12} {'σ²/(n-1)':>12} {'λ (PCA)':>12} {'Match?':>8}")
for i in range(5):
    lambda_from_svd = S[i]**2 / (n - 1)
    match = np.isclose(lambda_from_svd, eig_vals[i], rtol=1e-6)
    print(f"{i+1:>10} {S[i]:>12.4f} {lambda_from_svd:>12.4f} {eig_vals[i]:>12.4f} {'✓' if match else '✗':>8}")

# Compare principal directions
print(f"\nFirst PC from eigendecomposition: {eig_vecs[:3, 0].round(4)} ...")
print(f"First right singular vector:       {Vt[0, :3].round(4)} ...")
dot = abs(eig_vecs[:, 0] @ Vt[0])
print(f"Alignment (|dot product|): {dot:.6f} — identical direction.")

In [ ]:
# Truncated SVD for fast low-rank approximation
# Reconstruct digits using different numbers of singular values

sample_idx = 0  # a zero
ks = [1, 5, 10, 20, 40, 64]

fig, axes = plt.subplots(1, len(ks) + 1, figsize=(16, 2.5))

# Original
axes[0].imshow(images[sample_idx], cmap="gray_r")
axes[0].set_title("Original", fontsize=10)
axes[0].set_xticks([])
axes[0].set_yticks([])

for i, k in enumerate(ks):
    # Truncated SVD reconstruction
    X_recon = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]
    img_recon = (X_recon[sample_idx] + data.mean(axis=0)).reshape(8, 8)

    axes[i+1].imshow(img_recon, cmap="gray_r")
    error = np.mean((X[sample_idx] - X_recon[sample_idx])**2)
    axes[i+1].set_title(f"k={k} (err={error:.1f})", fontsize=10)
    axes[i+1].set_xticks([])
    axes[i+1].set_yticks([])

fig.suptitle("Truncated SVD: reconstruct a digit with increasing components",
             fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/03d_svd_reconstruction.png", dpi=150, bbox_inches="tight")
plt.show()

print("With ~10-20 components, the digit is clearly recognisable.")
print("SVD gives us efficient low-rank compression of the data matrix.")

**When to use SVD over eigendecomposition:**
- When $n < d$ (more features than samples) — SVD is more efficient
- When the data matrix is sparse — SVD handles this natively
- When you need a truncated approximation — `scipy.sparse.linalg.svds` computes only the top $k$ components
- For numerical stability — SVD avoids forming $X^TX$ explicitly, which can amplify rounding errors

## Factor Analysis

Factor analysis looks like PCA but makes a different assumption:

$$\mathbf{x} = W \mathbf{z} + \boldsymbol{\epsilon}$$

where $\mathbf{z}$ are latent factors, $W$ is the loading matrix, and $\boldsymbol{\epsilon}$ is **per-feature noise** with its own variance for each feature.

PCA treats all noise as equal. Factor analysis says: some features are noisier than others, and I want to model that.

In [ ]:
# Compare PCA and Factor Analysis on the digits
n_comp = 10

pca = PCA(n_components=n_comp)
X_pca = pca.fit_transform(data)

fa = FactorAnalysis(n_components=n_comp, random_state=42)
X_fa = fa.fit_transform(data)

# Compare loadings/components
fig, axes = plt.subplots(2, n_comp, figsize=(16, 4))

for i in range(n_comp):
    axes[0, i].imshow(pca.components_[i].reshape(8, 8), cmap="RdBu_r")
    axes[0, i].set_xticks([])
    axes[0, i].set_yticks([])
    if i == 0:
        axes[0, i].set_ylabel("PCA", fontsize=11, fontweight="bold")

    axes[1, i].imshow(fa.components_[i].reshape(8, 8), cmap="RdBu_r")
    axes[1, i].set_xticks([])
    axes[1, i].set_yticks([])
    if i == 0:
        axes[1, i].set_ylabel("FA", fontsize=11, fontweight="bold")

fig.suptitle("PCA vs Factor Analysis: component patterns (digits)", fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/03d_pca_vs_fa.png", dpi=150, bbox_inches="tight")
plt.show()

# Show per-feature noise variance from FA
fig, ax = plt.subplots(figsize=(10, 3))
ax.imshow(fa.noise_variance_.reshape(8, 8), cmap="hot")
ax.set_title("Factor Analysis: estimated noise variance per pixel")
ax.set_xticks([])
ax.set_yticks([])
plt.colorbar(ax.images[0], ax=ax, label="Noise variance")
plt.tight_layout()
plt.show()

print("FA estimates that corner/edge pixels have higher noise variance.")
print("This makes sense — corners are mostly blank, so variation there is noise.")

## ICA: Independence, Not Variance

PCA finds **uncorrelated** components (covariance = 0). ICA finds **statistically independent** components — a stronger condition. Independent means knowing one component tells you nothing about the others, including higher-order relationships.

The classic demo: the **cocktail party problem**. Two microphones record two speakers talking simultaneously. Each recording is a mixture of both voices. ICA unmixes them.

In [ ]:
# Cocktail party: generate 2 source signals, mix them, recover with ICA

# Source signals: one sine wave, one sawtooth
t = np.linspace(0, 1, 2000)
s1 = np.sin(2 * np.pi * 5 * t)          # 5 Hz sine
s2 = 2 * (t * 11 % 1) - 1               # 11 Hz sawtooth

S_sources = np.column_stack([s1, s2])

# Mixing matrix (unknown in practice)
A = np.array([[1.0, 0.7],
              [0.5, 1.0]])
X_mixed = S_sources @ A.T

# Recover with PCA and ICA
pca_mix = PCA(n_components=2)
X_pca_mix = pca_mix.fit_transform(X_mixed)

ica = FastICA(n_components=2, random_state=42)
X_ica = ica.fit_transform(X_mixed)

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)

# Original sources
axes[0].plot(t, s1, color=COLOURS["signal"], alpha=0.8, label="Source 1 (sine)")
axes[0].plot(t, s2, color=COLOURS["noise"], alpha=0.8, label="Source 2 (sawtooth)")
axes[0].set_title("Original sources (unknown)")
axes[0].legend(loc="upper right")

# Mixed signals
axes[1].plot(t, X_mixed[:, 0], color=COLOURS["neutral"], alpha=0.8, label="Mic 1")
axes[1].plot(t, X_mixed[:, 1], color="#8B5CF6", alpha=0.8, label="Mic 2")
axes[1].set_title("Mixed signals (what we observe)")
axes[1].legend(loc="upper right")

# PCA recovery
axes[2].plot(t, X_pca_mix[:, 0], color=COLOURS["signal"], alpha=0.8, label="PCA comp 1")
axes[2].plot(t, X_pca_mix[:, 1], color=COLOURS["noise"], alpha=0.8, label="PCA comp 2")
axes[2].set_title("PCA recovery — uncorrelated, but still mixed")
axes[2].legend(loc="upper right")

# ICA recovery
axes[3].plot(t, X_ica[:, 0], color=COLOURS["signal"], alpha=0.8, label="ICA comp 1")
axes[3].plot(t, X_ica[:, 1], color=COLOURS["noise"], alpha=0.8, label="ICA comp 2")
axes[3].set_title("ICA recovery — independent sources recovered")
axes[3].legend(loc="upper right")
axes[3].set_xlabel("Time")

fig.suptitle("Cocktail Party Problem: PCA vs ICA", fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/03d_cocktail_party.png", dpi=150, bbox_inches="tight")
plt.show()

print("PCA makes components uncorrelated but doesn't separate the sources.")
print("ICA finds components that are statistically independent — true source separation.")

In [ ]:
# Compare PCA and ICA on digits — different goals, different results
pca_digits = PCA(n_components=6)
ica_digits = FastICA(n_components=6, random_state=42)

X_pca_d = pca_digits.fit_transform(data)
X_ica_d = ica_digits.fit_transform(data)

fig, axes = plt.subplots(2, 6, figsize=(14, 4))

for i in range(6):
    axes[0, i].imshow(pca_digits.components_[i].reshape(8, 8), cmap="RdBu_r")
    axes[0, i].set_xticks([])
    axes[0, i].set_yticks([])
    if i == 0:
        axes[0, i].set_ylabel("PCA", fontsize=11, fontweight="bold")

    # ICA components (mixing matrix columns)
    comp = ica_digits.mixing_[:, i]
    axes[1, i].imshow(comp.reshape(8, 8), cmap="RdBu_r")
    axes[1, i].set_xticks([])
    axes[1, i].set_yticks([])
    if i == 0:
        axes[1, i].set_ylabel("ICA", fontsize=11, fontweight="bold")

fig.suptitle("PCA vs ICA components on digits — different decompositions",
             fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/03d_pca_vs_ica_digits.png", dpi=150, bbox_inches="tight")
plt.show()

print("PCA components are ordered by variance — global patterns first.")
print("ICA components are more localised — they capture independent spatial patterns.")

## When to Use What

| Method | Assumption | Goal | Use When |
|--------|-----------|------|----------|
| **PCA** | Linear, variance matters | Max variance directions | Default choice. Exploration, preprocessing, denoising |
| **SVD** | Same as PCA | Same as PCA | Large/sparse matrices, numerical stability, truncated computation |
| **Factor Analysis** | Latent factors + per-feature noise | Latent structure | You believe in a generative model with noisy measurements |
| **ICA** | Sources are statistically independent | Source separation | Signal unmixing (fMRI, audio, EEG) |

**Rules of thumb:**
- **Start with PCA.** It's fast, well-understood, and works as a baseline.
- **Use SVD** when PCA's eigendecomposition is too slow or numerically unstable (very wide matrices).
- **Use Factor Analysis** when you care about the generative model and want per-feature noise estimates.
- **Use ICA** when you need to separate mixed signals, not just decorrelate them.

<details>
<summary><b>The Maths Behind This</b></summary>

**SVD-PCA connection:** For centred data $\bar{X}$ with SVD $\bar{X} = U\Sigma V^T$:
- PCA eigenvalues: $\lambda_i = \sigma_i^2 / (n-1)$
- PCA eigenvectors = columns of $V$ (right singular vectors)
- PCA scores = $\bar{X}V = U\Sigma$ (left singular vectors scaled by singular values)

**Factor Analysis model:** $\mathbf{x} = W\mathbf{z} + \boldsymbol{\mu} + \boldsymbol{\epsilon}$
- $\mathbf{z} \sim \mathcal{N}(0, I)$ — latent factors
- $\boldsymbol{\epsilon} \sim \mathcal{N}(0, \Psi)$ where $\Psi = \text{diag}(\psi_1, \ldots, \psi_d)$
- Implied covariance: $\Sigma = WW^T + \Psi$
- PCA is the special case where $\Psi = \sigma^2 I$ (equal noise everywhere)

**ICA model:** $\mathbf{x} = A\mathbf{s}$ where components of $\mathbf{s}$ are:
- Mutually independent (not just uncorrelated)
- Non-Gaussian (Gaussians can't be separated — a key limitation)
- ICA maximises non-Gaussianity of the recovered components

</details>

## Where This Matters

**Medical imaging (fMRI):** Brain scans produce time series at thousands of voxels. ICA separates spatially independent brain networks from the mixed signal — revealing which brain regions activate together during a task.

**Recommender systems:** Netflix's recommendation engine used SVD on the user-movie rating matrix (very sparse, very large). Truncated SVD extracted latent factors (genres, moods) without ever forming the full covariance matrix.

**Psychometrics:** Factor analysis was literally invented for this — extracting latent psychological traits (intelligence, personality dimensions) from noisy survey responses.

## Feynman Check

1. **What's the relationship between SVD and PCA?**

2. **When would you choose ICA over PCA?**

3. **What assumption does factor analysis make that PCA doesn't?**

---

Module 03 is complete. You now have a solid understanding of linear dimensionality reduction — from the geometric intuition of projections (03a), through building PCA from scratch (03b), applying it to real data (03c), and understanding its siblings (03d).

But linear methods have a fundamental limitation: they can't handle curved structure. In **Module 04: Nonlinear Methods**, we'll tackle t-SNE, UMAP, Isomap, and other methods that can follow the data wherever it goes.